In [1]:
# =====================================================================
# MACHINE LEARNING (ASSIGNMENT 2)
# PART 1: DATA PIPELINE, COHORT TRAINING AND SERIALIZATION
# =====================================================================

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# --- 1. DATA INGESTION & HEADER MAPPING ---
# Ensure your uploaded file in Colab is named exactly 'test_data.csv'
dataset_path = 'test_data.csv'
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Missing resource file: {dataset_path}. Please upload it via the sidebar.")

raw_df = pd.read_csv(dataset_path, header=None)

# Explicitly define clean features as requested by the clinical domain metadata
feature_headers = [
    'sample_id', 'target_diagnosis', 'radius_avg', 'texture_avg', 'perimeter_avg', 'area_avg',
    'smoothness_avg', 'compactness_avg', 'concavity_avg', 'concave_pts_avg', 'symmetry_avg',
    'fractal_dim_avg', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
    'compactness_se', 'concavity_se', 'concave_pts_se', 'symmetry_se', 'fractal_dim_se',
    'radius_extreme', 'texture_high', 'perimeter_extreme', 'area_extreme', 'smoothness_extreme',
    'compactness_extreme', 'concavity_extreme', 'concave_pts_extreme', 'symmetry_extreme', 'fractal_dim_extreme'
]
raw_df.columns = feature_headers

# Drop the indexing primary key
refined_df = raw_df.drop(columns=['sample_id'])

# --- 2. TARGET ENCODING & SPLITTING ---
encoder = LabelEncoder()
refined_df['target_diagnosis'] = encoder.fit_transform(refined_df['target_diagnosis']) # Malignant -> 1, Benign -> 0

X = refined_df.drop(columns=['target_diagnosis'])
y = refined_df['target_diagnosis']

# Perform a stratified 80-20 train-test split to maintain target ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=101, stratify=y
)

# --- 3. STANDARD NORMALIZATION ---
# Algorithms like KNN, Logistic Regression, and SVM require feature standardization
normalizer = StandardScaler()
X_train_scaled = normalizer.fit_transform(X_train)
X_test_scaled = normalizer.transform(X_test)

# Create structural output directories
os.makedirs('model', exist_ok=True)

# Export validation pipeline dependencies
joblib.dump(normalizer, 'model/scaler.pkl')
joblib.dump(encoder, 'model/label_encoder.pkl')

# Save partitioned unscaled features along with targets as 'test_data.csv' for Streamlit use
streamlit_test_payload = X_test.copy()
streamlit_test_payload['diagnosis'] = y_test.values
streamlit_test_payload.to_csv('model/test_data.csv', index=False)

# --- 4. ALGORITHM ENGINE CORTEGE ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

algorithm_portfolio = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='lbfgs', random_state=101),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=101),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='uniform'),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(C=1.0, kernel='rbf', probability=True, random_state=101)
}

# --- 5. EVALUATION AND ARTIFACT GENERATION ---
matrix_summary = []

print("\n" + "="*60)
print("             EVALUATION GENERATED PERFORMANCE METRICS       ")
print("="*60)

for engine_name, engine_object in algorithm_portfolio.items():
    # Route data transformation conditions appropriately based on algorithm class
    if engine_name in ['Decision Tree', 'Naive Bayes']:
        engine_object.fit(X_train, y_train)
        predictions = engine_object.predict(X_test)
        prob_scores = engine_object.predict_proba(X_test)[:, 1]
    else:
        engine_object.fit(X_train_scaled, y_train)
        predictions = engine_object.predict(X_test_scaled)
        prob_scores = engine_object.predict_proba(X_test_scaled)[:, 1]

    # Save model binary file into target directory path
    safe_filename = f"model/{engine_name.lower().replace(' ', '_')}_classifier.pkl"
    joblib.dump(engine_object, safe_filename)

    # Calculate validation indices
    acc  = accuracy_score(y_test, predictions)
    auc  = roc_auc_score(y_test, prob_scores)
    prec = precision_score(y_test, predictions)
    rec  = recall_score(y_test, predictions)
    f1   = f1_score(y_test, predictions)
    mcc  = matthews_corrcoef(y_test, predictions)

    matrix_summary.append({
        'ML Model Name': engine_name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

# Format results as a clear markdown summary table
summary_view_df = pd.DataFrame(matrix_summary)
print(summary_view_df.to_markdown(index=False))
print("="*60 + "\nExecution Complete.")



             EVALUATION GENERATED PERFORMANCE METRICS       
| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |     0.9649 | 0.9917 |      0.975  |   0.9286 | 0.9512 | 0.9245 |
| Decision Tree          |     0.886  | 0.8887 |      0.8372 |   0.8571 | 0.8471 | 0.7563 |
| KNN                    |     0.9474 | 0.9798 |      0.9737 |   0.881  | 0.925  | 0.8872 |
| Naive Bayes            |     0.9386 | 0.9848 |      0.9487 |   0.881  | 0.9136 | 0.8675 |
| Support Vector Machine |     0.9474 | 0.9954 |      0.9286 |   0.9286 | 0.9286 | 0.8869 |
Execution Complete.
